# FFmpeg Processing Plugin

> FFmpeg-based media processing plugin implementing the `MediaProcessingPlugin` interface.

In [ ]:
#| default_exp plugin

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
import logging
import os
import shutil
import time
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

from cjm_media_plugin_system.processing_interface import MediaProcessingPlugin
from cjm_media_plugin_system.core import MediaMetadata
from cjm_media_plugin_system.storage import MediaProcessingStorage

from cjm_plugin_system.utils.hashing import hash_file
from cjm_plugin_system.utils.validation import (
    dict_to_config, config_to_dict, dataclass_to_jsonschema,
    SCHEMA_TITLE, SCHEMA_DESC, SCHEMA_ENUM
)

from cjm_ffmpeg_utils.core import FFMPEG_AVAILABLE, get_audio_codec
from cjm_ffmpeg_utils.media_info import get_audio_info_ffmpeg, get_media_duration
from cjm_ffmpeg_utils.audio import extract_audio_segment, convert_to_mp3
from cjm_ffmpeg_utils.execution import run_ffmpeg_with_progress

from cjm_media_plugin_ffmpeg.meta import get_plugin_metadata

## Configuration

In [ ]:
#| export
@dataclass
class FFmpegPluginConfig:
    """Configuration for the FFmpeg processing plugin."""

    output_dir: Optional[str] = field(
        default=None,
        metadata={
            SCHEMA_TITLE: "Output Directory",
            SCHEMA_DESC: "Default output directory for processed files. Uses plugin data_dir if None."
        }
    )

    default_audio_format: str = field(
        default="mp3",
        metadata={
            SCHEMA_TITLE: "Default Audio Format",
            SCHEMA_DESC: "Default format for audio extraction and conversion.",
            SCHEMA_ENUM: ["mp3", "wav", "flac", "aac", "ogg", "m4a"]
        }
    )

    default_audio_bitrate: str = field(
        default="192k",
        metadata={
            SCHEMA_TITLE: "Default Audio Bitrate",
            SCHEMA_DESC: "Default bitrate for audio encoding operations.",
            SCHEMA_ENUM: ["128k", "192k", "256k", "320k"]
        }
    )

    prefer_stream_copy: bool = field(
        default=True,
        metadata={
            SCHEMA_TITLE: "Prefer Stream Copy",
            SCHEMA_DESC: "When possible, copy audio stream without re-encoding (faster, lossless)."
        }
    )

In [ ]:
schema = dataclass_to_jsonschema(FFmpegPluginConfig)
assert "properties" in schema
assert "default_audio_format" in schema["properties"]
assert schema["properties"]["default_audio_format"]["default"] == "mp3"
print(f"Config schema: {list(schema['properties'].keys())}")

Config schema: ['output_dir', 'default_audio_format', 'default_audio_bitrate', 'prefer_stream_copy']


## Plugin Class

In [ ]:
#| export
class FFmpegProcessingPlugin(MediaProcessingPlugin):
    """FFmpeg-based media processing plugin."""

    config_class = FFmpegPluginConfig

    def __init__(self):
        """Initialize the FFmpeg processing plugin."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: Optional[FFmpegPluginConfig] = None
        self.storage: Optional[MediaProcessingStorage] = None
        self._data_dir: Optional[str] = None

    @property
    def name(self) -> str:  # Plugin identifier
        return "ffmpeg"

    @property
    def version(self) -> str:  # Plugin version
        return "1.0.0"

    @property
    def supported_media_types(self) -> List[str]:  # Supported input types
        return ["audio", "video"]

    def initialize(self, config: Optional[Any] = None) -> None:
        """Initialize plugin with configuration."""
        self.config = dict_to_config(FFmpegPluginConfig, config or {})
        meta = get_plugin_metadata()
        db_path = meta["db_path"]
        self._data_dir = os.path.dirname(db_path)
        self.storage = MediaProcessingStorage(db_path)
        self.logger.info(f"Initialized FFmpeg plugin (format={self.config.default_audio_format})")

    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for UI form generation
        """Return the JSON Schema for plugin configuration."""
        return dataclass_to_jsonschema(FFmpegPluginConfig)

    def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
        """Return the current configuration as a dictionary."""
        return config_to_dict(self.config) if self.config else {}

    def is_available(self) -> bool:  # Whether ffmpeg is installed
        """Check if ffmpeg is available on this system."""
        return FFMPEG_AVAILABLE

    def cleanup(self) -> None:
        """Clean up plugin resources."""
        self.logger.info("FFmpeg plugin cleaned up")

    # ------------------------------------------------------------------
    # Action dispatch
    # ------------------------------------------------------------------

    def execute(self,
                action: str = "get_info",  # Action to perform
                **kwargs
               ) -> Dict[str, Any]:  # Action result
        """Dispatch to the appropriate action handler."""
        if action == "get_info":
            return self.get_info(kwargs["file_path"]).to_dict()
        elif action == "convert":
            output = self.convert(kwargs["input_path"], kwargs["output_format"], **{
                k: v for k, v in kwargs.items() if k not in ("input_path", "output_format")
            })
            return {"output_path": output}
        elif action == "extract_segment":
            output = self.extract_segment(
                kwargs["input_path"], kwargs["start"], kwargs["end"],
                kwargs.get("output_path")
            )
            return {"output_path": output}
        elif action == "extract_audio":
            return self._extract_audio(**kwargs)
        elif action == "segment_audio":
            return self._segment_audio(**kwargs)
        else:
            raise ValueError(f"Unknown action: {action}")

    # ------------------------------------------------------------------
    # Required abstract methods (stubs for Phase 2)
    # ------------------------------------------------------------------

    def get_info(self,
                 file_path: Union[str, Path],  # Path to media file
                ) -> MediaMetadata:  # Probed metadata
        """Get metadata for a media file via ffprobe."""
        raise NotImplementedError("get_info will be implemented in Phase 2")

    def convert(self,
                input_path: Union[str, Path],  # Source file path
                output_format: str,  # Target format (e.g. 'mp3', 'wav')
                **kwargs
               ) -> str:  # Output file path
        """Convert media to a different format."""
        raise NotImplementedError("convert will be implemented in Phase 2")

    def extract_segment(self,
                        input_path: Union[str, Path],  # Source audio file
                        start: float,  # Start time in seconds
                        end: float,  # End time in seconds
                        output_path: Optional[str] = None,  # Custom output path
                       ) -> str:  # Output file path
        """Extract a temporal segment from a media file."""
        raise NotImplementedError("extract_segment will be implemented in Phase 2")

    # ------------------------------------------------------------------
    # Custom actions (stubs for Phase 3)
    # ------------------------------------------------------------------

    def _extract_audio(self, **kwargs) -> Dict[str, Any]:
        """Extract audio stream from a video file."""
        raise NotImplementedError("_extract_audio will be implemented in Phase 3")

    def _segment_audio(self, **kwargs) -> Dict[str, Any]:
        """Split audio file into segments at specified boundaries."""
        raise NotImplementedError("_segment_audio will be implemented in Phase 3")

In [ ]:
plugin = FFmpegProcessingPlugin()
assert plugin.name == "ffmpeg"
assert plugin.version == "1.0.0"
assert plugin.supported_media_types == ["audio", "video"]
assert plugin.is_available() == FFMPEG_AVAILABLE
print(f"Plugin: {plugin.name} v{plugin.version}")
print(f"FFmpeg available: {plugin.is_available()}")

Plugin: ffmpeg v1.0.0
FFmpeg available: True


In [ ]:
plugin.initialize({})
assert plugin.config is not None
assert plugin.config.default_audio_format == "mp3"
assert plugin.config.prefer_stream_copy is True
assert plugin.storage is not None
print(f"Config: format={plugin.config.default_audio_format}, bitrate={plugin.config.default_audio_bitrate}")
print(f"Current config: {plugin.get_current_config()}")

Config: format=mp3, bitrate=192k
Current config: {'output_dir': None, 'default_audio_format': 'mp3', 'default_audio_bitrate': '192k', 'prefer_stream_copy': True}


In [ ]:
plugin.initialize({"default_audio_format": "wav", "default_audio_bitrate": "320k"})
assert plugin.config.default_audio_format == "wav"
assert plugin.config.default_audio_bitrate == "320k"
print(f"Custom config: {plugin.get_current_config()}")

Custom config: {'output_dir': None, 'default_audio_format': 'wav', 'default_audio_bitrate': '320k', 'prefer_stream_copy': True}


In [ ]:
schema = plugin.get_config_schema()
assert "properties" in schema
props = schema["properties"]
assert "output_dir" in props
assert "default_audio_format" in props
assert "default_audio_bitrate" in props
assert "prefer_stream_copy" in props
print(f"Schema properties: {list(props.keys())}")

Schema properties: ['output_dir', 'default_audio_format', 'default_audio_bitrate', 'prefer_stream_copy']


In [ ]:
plugin.cleanup()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()